# shotforge — Colab 流水线（CLI 优先）

**推荐用法**：跑完下面的 Setup 后，打开 Colab 的 **Terminal**，全程用 `python run.py ...`。
下面的 cell 只是等效便利 + 看图/看片预览。

**先做**：`Runtime → Change runtime type → GPU`（A100 最佳）；左侧 🔑 Secrets 加 `HF_TOKEN`(可选,加速)。


## 1) Setup（一次性）：ComfyUI + Wan(视频) + Kontext(分镜) + SDXL动漫(参考图)


In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get('HF_TOKEN')
    if _t:
        os.environ['HF_TOKEN'] = _t; print('HF_TOKEN loaded')
except Exception:
    print('no HF_TOKEN (ok — models are public, just slower)')

!git clone https://github.com/chengh233/shotforge /content/shotforge 2>/dev/null; cd /content/shotforge && git pull -q
!cd /content/shotforge && python scripts/colab_setup.py    # ComfyUI + Wan I2V 模型
!cd /content/shotforge && python scripts/flux_setup.py     # Flux Kontext + Animagine(SDXL动漫)


## 2) 开隧道 + 导出两个工作流（一次性手动）
Kontext(分镜) 和 SDXL(参考图) 需要各导出一次 API 工作流。开隧道从 Mac 浏览器进 ComfyUI：
- Browse Templates → **SDXL** txt2img → Dev mode → Export(API) → 存 `comfyui/sdxl_txt2img_api.json`
- Browse Templates → **Flux Kontext** → Dev mode → Export(API) → 存 `comfyui/flux_kontext_api.json`
（放进 `/content/shotforge/comfyui/`。视频 Wan 工作流已在仓库里。）


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!cloudflared tunnel --url http://localhost:8188   # 打印 https://xxx.trycloudflare.com，Mac 浏览器打开


## 3) 跑流水线 —— **推荐在 Terminal 里用 CLI**（下面 cell 等效）
```bash
python run.py genref yuki                                 # 角色库 yuki 出参考图(SDXL)
python run.py frames projects/lasttram                    # cast 的 yuki 一键出全部分镜帧(Kontext)
python run.py video  projects/lasttram --shot s1          # 先单镜
python run.py video  projects/lasttram                    # 全部镜头
python run.py dub    projects/lasttram                    # 用 yuki 的声音配音
python run.py subs   projects/lasttram
python run.py post   projects/lasttram --crossfade 0.5 --fade 0.6   # 溶解+淡入淡出
```


### （等效 cell）出参考图 + 分镜帧


In [ ]:
!cd /content/shotforge && python run.py genref yuki
!cd /content/shotforge && python run.py frames projects/lasttram


### 看分镜帧（质量门）


In [ ]:
from IPython.display import Image, display
for s in ['s1','s2','s3','s4','s5','s6']:
    print(s)
    display(Image(f'/content/shotforge/projects/lasttram/frames/{s}.jpeg', width=220))


### 渲染单镜 → 看


In [ ]:
!cd /content/shotforge && python run.py video projects/lasttram --shot s1


In [ ]:
from IPython.display import Video
Video('/content/shotforge/projects/lasttram/out/s1.mp4', embed=True, width=300)


### 全部镜头 → 配音 → 字幕 → 合成


In [ ]:
!cd /content/shotforge && python run.py video projects/lasttram
!cd /content/shotforge && python run.py dub  projects/lasttram
!cd /content/shotforge && python run.py subs projects/lasttram
!cd /content/shotforge && python run.py post projects/lasttram --crossfade 0.5 --fade 0.6


### 看成片


In [ ]:
from IPython.display import Video
Video('/content/shotforge/projects/lasttram/out/末班电车_final.mp4', embed=True, width=300)
